# turboquant-gpu

5.02x KV cache compression for LLM inference using cuTile kernels with PyTorch fallback.

Paper: [TurboQuant](https://arxiv.org/abs/2501.09747) (ICLR 2026)

## install

Detects your CUDA driver version and installs a compatible PyTorch wheel. Also installs `cuda-tile` for GPU kernel acceleration (falls back to PyTorch if unavailable). Restart the kernel after this cell finishes.

In [ ]:
import subprocess, sys, ctypes

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", *args], check=True)

subprocess.run([sys.executable, "-m", "ensurepip", "--upgrade"],
               capture_output=True)

try:
    libcuda = ctypes.CDLL("libcuda.so.1")
    v = ctypes.c_int()
    libcuda.cuInit(0)
    libcuda.cuDriverGetVersion(ctypes.byref(v))
    drv = v.value
except Exception:
    drv = 0

if   drv >= 12800: whl = "cu128"
elif drv >= 12400: whl = "cu124"
elif drv >= 12100: whl = "cu121"
else:              whl = "cu118"

print(f"CUDA driver {drv} -> pytorch {whl}")
pip("torch", "--index-url", f"https://download.pytorch.org/whl/{whl}", "--force-reinstall")

try:
    pip("cuda-tile[tileiras]", "--extra-index-url", "https://pypi.nvidia.com")
except Exception:
    print("cuda-tile not available, will use pytorch fallback")

pip("turboquant-gpu")
pip("scipy", "transformers", "accelerate", "sentencepiece")
print("\ndone. restart kernel then run next cell.")

## setup

Import the engine and detect the GPU. If no CUDA device is found, everything runs on CPU with PyTorch fallback.

In [ ]:
import torch, time
from turboquant_gpu import TurboQuantEngine
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"

if device == "cuda":
    gpu = torch.cuda.get_device_name()
    sm  = torch.cuda.get_device_capability()
    print(f"{gpu}  |  sm_{sm[0]}{sm[1]}  |  CUDA {torch.version.cuda}")
else:
    print("no GPU, running on CPU")

## load model

Load Mistral 7B in FP16. The `TurboQuantEngine` takes `head_dim` (128 for Mistral) and `total_bits=3` which gives 2-bit key MSE + 1-bit QJL correction, and 3-bit value quantization.

In [ ]:
model_id = "mistralai/Mistral-7B-v0.1"

tok   = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, torch_dtype=torch.float16, device_map=device)

head_dim = model.config.hidden_size // model.config.num_attention_heads
engine   = TurboQuantEngine(head_dim=head_dim, total_bits=3, device=device)
print(f"head_dim={head_dim}  |  device={device}")

## generate with compressed KV cache

One call does everything: prefill, compress the KV cache to 3 bits, rebuild a `DynamicCache`, and decode autoregressively. The compression ratio and generation time are printed at the end.

In [ ]:
t0 = time.time()
out = engine.generate(model, tok, "The University of Waterloo is known for ")
t1 = time.time()

print(out["text"])
print(f"\n{out['tokens']} tokens  |  {out['stats']['ratio']:.2f}x compression  |  {t1-t0:.2f}s")

## auto-tune

Benchmarks all combinations of bit-width (2 vs 3) and backend (cuTile vs PyTorch) on your specific GPU. Picks the fastest configuration that passes a cosine similarity quality threshold. After this, the engine uses the winning config for all subsequent calls.

In [ ]:
engine.auto_tune(seq_len=512)

## generate again (after auto-tune)

Same prompt, same model, but now using whatever config auto-tune selected. Compare the timing and compression ratio to the run above.

In [ ]:
t0 = time.time()
out = engine.generate(model, tok, "The University of Waterloo is known for ")
t1 = time.time()

print(out["text"])
print(f"\n{out['tokens']} tokens  |  {out['stats']['ratio']:.2f}x compression  |  {t1-t0:.2f}s (after auto-tune)")